# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdeenMir/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import duckdb, os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('Adeen')
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Unit of analysis + time window

One row = one client's content item, on one day (report_date × client_hash_id × content_hash_id), in fact_content_daily_performance. Time window: the month=2026-03 partition — a mid-panel month, not the sealed final-month _sample.

In [6]:
con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



## 2. Fields: feature / label / context / excluded

Feature: gsc_avg_position, gsc_impressions, gsc_clicks, ga4_engaged_sessions, scroll_events — all daily-observed values, knowable at the decision moment because they're logged for days that already happened.
Label/proxy: decline in gsc_clicks from the first half of the month to the second half — never used as a feature.
Context (join/filter only, never features): client_hash_id, content_hash_id (pseudonymous IDs), month (partition key).
Excluded: client_has_gsc/client_has_ga4 — static account-level flags, not day-level signal, risk of leaking "this client is a bigger/more mature account" rather than content-level behavior; ai_chatgpt/ai_perplexity/etc. — too sparse and out of scope for this lane, excluded for simplicity, not correctness.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
con.sql(f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS first_day, MAX(report_date) AS last_day
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").show()
df = con.sql(f"""
SELECT
  content_hash_id, client_hash_id,
  AVG(CASE WHEN day(report_date) <= 15 THEN gsc_avg_position END) AS avg_position_h1,
  AVG(CASE WHEN day(report_date) <= 15 THEN gsc_impressions END) AS impressions_h1,
  AVG(CASE WHEN day(report_date) <= 15 THEN gsc_clicks END) AS clicks_h1,
  AVG(CASE WHEN day(report_date) <= 15 THEN ga4_engaged_sessions END) AS engaged_sessions_h1,
  AVG(CASE WHEN day(report_date) <= 15 THEN scroll_events END) AS scroll_events_h1,
  AVG(CASE WHEN day(report_date) > 15 THEN gsc_clicks END) AS clicks_h2
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE
GROUP BY content_hash_id, client_hash_id
HAVING COUNT(*) > 10
""").df()

df['is_declining'] = (df['clicks_h2'] < df['clicks_h1'] * 0.8).astype(int)
df = df.dropna()

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_honest = df[['avg_position_h1','impressions_h1','clicks_h1','engaged_sessions_h1','scroll_events_h1']].fillna(0)
y = df['is_declining']

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:,1])
print("Honest AUC:", honest_auc)

# THE TRAP: add a column derived straight from the label
X_leaky = X_honest.copy()
X_leaky['clicks_h2_LEAK'] = df['clicks_h2']  # this literally defines is_declining

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.3, random_state=42)
clf_leaky = LogisticRegression(max_iter=1000).fit(X_train, y_train)
leaky_auc = roc_auc_score(y_test, clf_leaky.predict_proba(X_test)[:,1])
print("Leaky AUC (should jump toward 1.0):", leaky_auc)

del X_leaky
print("Final honest AUC kept:", honest_auc)


┌───────────┬────────────┬────────────┐
│ row_count │ first_day  │  last_day  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC: 0.7357470939499813
Leaky AUC (should jump toward 1.0): 0.9993700251943788
Final honest AUC kept: 0.7357470939499813


## 4. Data limits

This slice only reflects rows where gsc_data_available IS TRUE; clients without GSC hooked up, or with data starting later, are underrepresented, so this ranking favors longer-tenured clients over newly onboarded ones.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.